In [1]:
import sys
sys.dont_write_bytecode = True

import warnings
warnings.filterwarnings("ignore")

import importlib
from src.main.python.schema import model
importlib.reload(model)

<module 'src.main.python.schema.model' from 'd:\\dev_space\\LLM-Server\\src\\main\\python\\schema\\model.py'>

# unittest

In [2]:
import sys
import subprocess
!python -B -m unittest src.unittest.python.test_LLMserver

.
----------------------------------------------------------------------
Ran 1 test in 0.082s

OK


# dev

In [1]:
import sys
sys.dont_write_bytecode = True

from src.main.python.engine import decode, prefill
from src.main.python.scheduler import scheduler
from src.main.python.config import config
from src.main.python.schema import model
import importlib

import torch
from transformers import GemmaTokenizerFast, BitsAndBytesConfig, Gemma3ForCausalLM, DynamicCache
PATH = "D://LLM//gemma//gemma3_4b"
# PATH = "D://LLM//small_gemma//gemma3_270M"

quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

llm_model = Gemma3ForCausalLM.from_pretrained(
    PATH,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True
    )
llm_model = llm_model.eval()
tokenizer = GemmaTokenizerFast.from_pretrained(PATH)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
import uuid
import torch
import importlib
# importlib.reload(scheduler)
MSG = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""
TEXT = dict()
_scheduler = scheduler.RequestManager()

sentences = ("半導體廠務通常在做什麼", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理，它裡面有很多關於統計觀念，也請一一闡述", "什麼是巨單交易", "什麼是機器學習")
for i in range(8):
    if i < 4:
        sentence = sentences[i]
    ids = tokenizer.encode(MSG.format(prompt=sentence))
    request = model.Request(input_ids = torch.tensor(ids).unsqueeze(0),
                            status = model.RequestStatus.PREFILLING,
                            request_id = str(uuid.uuid4()),
                            kv_cache = DynamicCache()
            )
    # print("token長度:", len(ids))
    _scheduler.add_request(request)

    d_input_ids, d_caches, p_input_ids, p_caches, decode_requests = _scheduler.step()
    text, DCACHES = decode.infer(llm_model, d_input_ids, d_caches, [r.request_id for r in decode_requests])
    _, PCACHES = prefill.infer(llm_model, p_input_ids, p_caches)
    _scheduler.update(PCACHES)
    if text is not None:
        for i, r in enumerate(decode_requests):
            _id = r.request_id
            r.input_ids = torch.tensor(text[_id][-1:]).unsqueeze(0)
            r.kv_cache = DCACHES[i]
            _scheduler.add_request(r)
            TEXT[_id] = TEXT.get(_id, "") + tokenizer.decode(text[_id], skip_special_tokens=True)
    print("----- 最後輸出 -----\n", TEXT, "\n", "-"*50)

----- 最後輸出 -----
 {} 
 --------------------------------------------------
----- 最後輸出 -----
 {} 
 --------------------------------------------------
----- 最後輸出 -----
 {} 
 --------------------------------------------------
----- 最後輸出 -----
 {'6a2c4ea6-bb22-4c4c-a296-a356d59ca732': '半導體廠務主要做的事情非常複雜，可以概括為以下'} 
 --------------------------------------------------
----- 最後輸出 -----
 {'6a2c4ea6-bb22-4c4c-a296-a356d59ca732': '半導體廠務主要做的事情非常複雜，可以概括為以下幾個關鍵步驟，大致可以分為以下幾個階段：\n\n**1. 設計'} 
 --------------------------------------------------
----- 最後輸出 -----
 {'6a2c4ea6-bb22-4c4c-a296-a356d59ca732': '半導體廠務主要做的事情非常複雜，可以概括為以下幾個關鍵步驟，大致可以分為以下幾個階段：\n\n**1. 設計與製程規劃 (Design & Process Planning):**\n\n* **晶片', 'fb080bd4-3116-4887-b79c-4baa7d235eb9': ''} 
 --------------------------------------------------
----- 最後輸出 -----
 {'6a2c4ea6-bb22-4c4c-a296-a356d59ca732': '半導體廠務主要做的事情非常複雜，可以概括為以下幾個關鍵步驟，大致可以分為以下幾個階段：\n\n**1. 設計與製程規劃 (Design & Process Planning):**\n\n* **晶片設計 (Chip Design):**  這是最核心的部分，由設計工程師', 'fb080bd4-

In [ ]:
import uuid
import torch
import importlib
# importlib.reload(scheduler)
MSG = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""
TEXT = dict()
_scheduler = scheduler.RequestManager()
for sentences in ("半導體廠務通常在做什麼", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理，它裡面有很多關於統計觀念，也請一一闡述", "什麼是巨單交易", "什麼是機器學習"):
    ids = tokenizer.encode(MSG.format(prompt=sentences))
    request = model.Request(input_ids = torch.tensor(ids).unsqueeze(0),
                            status = model.RequestStatus.PREFILLING,
                            request_id = str(uuid.uuid4()),
                            kv_cache = DynamicCache()
            )
    # print("token長度:", len(ids))
    _scheduler.add_request(request)

    for _ in range(8):
        d_input_ids, d_caches, p_input_ids, p_caches, decode_requests = _scheduler.step()
        text, DCACHES = decode.infer(llm_model, d_input_ids, d_caches, [r.request_id for r in decode_requests])
        _, PCACHES = prefill.infer(llm_model, p_input_ids, p_caches)
        _scheduler.update(PCACHES)
        if text is not None:
            for i, r in enumerate(decode_requests):
                _id = r.request_id
                r.input_ids = torch.tensor(text[_id][-1:]).unsqueeze(0)
                print(_id, tokenizer.decode(text[_id][:]))
                r.kv_cache = DCACHES[i]
                _scheduler.add_request(r)
                TEXT[_id] = TEXT.get(_id, "") + tokenizer.decode(text[_id], skip_special_tokens=True)
    print("----- 最後輸出 -----\n", TEXT, "\n", "-"*50)

4cde6fa8-6705-4e6f-a7d5-0cae56672940 半導體廠務主要做的事情非常複雜，可以概括為以下
4cde6fa8-6705-4e6f-a7d5-0cae56672940 幾個關鍵步驟，大致可以分為以下幾個階段：

**1. 設計
4cde6fa8-6705-4e6f-a7d5-0cae56672940 與製程規劃 (Design & Process Planning):**

* **晶片
4cde6fa8-6705-4e6f-a7d5-0cae56672940 設計 (Chip Design):**  這是最核心的部分，由設計工程師
4cde6fa8-6705-4e6f-a7d5-0cae56672940 使用專業軟體（例如Cadence、Synopsys）設計晶片
----- 最後輸出 -----
 {'4cde6fa8-6705-4e6f-a7d5-0cae56672940': '半導體廠務主要做的事情非常複雜，可以概括為以下幾個關鍵步驟，大致可以分為以下幾個階段：\n\n**1. 設計與製程規劃 (Design & Process Planning):**\n\n* **晶片設計 (Chip Design):**  這是最核心的部分，由設計工程師使用專業軟體（例如Cadence、Synopsys）設計晶片'} 
 --------------------------------------------------


KeyboardInterrupt: 

In [14]:
p_input_ids[0].device

device(type='cpu')

In [6]:
d_caches

[]

In [5]:
for t in TEXT:
    print(t, "\n", TEXT[t], "\n","-"*50,"\n")

d49313aa-5ebe-4abb-9753-1c7fee05fa47 
 半導體廠務主要做的事情非常複雜，可以概括為以下幾個關鍵步驟，大致可以分為以下幾個階段：

**1. 設計與製程規劃 (Design & Process Planning):**

* **晶片設計 (Chip Design):**  這是最核心的部分，由設計工程師使用專業軟體（例如Cadence、Synopsys）設計晶片內部電路的結構，包括邏輯電路、記憶體、處理器等等。
* **製程規劃 (Process Planning):**  根據設計，工程師會制定詳細的製造流程，包括使用的材料、設備、參數設定等，確保晶片能夠按照設計正確地製造出來。
* **模擬與驗證 (Simulation & Verification):**  在實際製造之前，會使用模擬軟體驗證設計的正確性，並預測晶片在實際製造中的表現。


**2. 製造 (Fabrication - 晶圓製造):**

這是半導體廠務最複雜、最昂貴的部分，主要包含以下幾個階段：

* **晶圓切割 (Wafer Fabrication):**  使用高純度的矽晶圓作為基底，進行切割。
* **薄膜堆疊 (Thin Film Deposition):**  利用各種技術（例如化學氣相沉積、物理氣相沉積）在晶圓上 depositing 不同的薄膜材料，形成電路元件。
* **光刻 (Photolithography):**  使用光刻機將設計圖案轉印到晶圓上，這是製造複雜電路的關鍵步驟。
* **蝕刻 (Etching):**  利用化學或物理方法去除晶圓上不需要的部分，形成電路圖案。
* **金屬化 (Metallization):**  在晶圓上 depositing 金屬層，連接不同的電路元件。
* **測試 (Testing):**  在晶圓上進行電氣測試，驗證每個電路元件的功能是否正常。


**3. 測試與封裝 (Testing & Packaging):**

* **晶圓測試 (Wafer Probe):**  使用自動測試設備（ATE）對晶圓上的每個電路進行測試，找出缺陷 
 -------------------------------------------------- 

b0ee6598-c880-43e8-bd

# test

In [19]:
msg = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""

def gemma3_resp(prompt, max_seq_len):

    # ----- 結果儲存 ----- #
    res = list()

    # ----- Prompt token產生 ----- #
    MSG = msg.format(prompt=prompt)
    input_ids = torch.tensor(tokenizer.encode(MSG)).to(llm_model.device)
    input_ids = input_ids.unsqueeze(0)
    eos_token_ids = [tokenizer.eos_token_id, 106]

    # ----- Cache宣告 ----- #
    past_key_values = DynamicCache()
    
    # ----- Prefill ----- #
    chunks = torch.split(input_ids[:, :-1], 32, dim=-1)
    st = 0
    ed = 0
    with torch.no_grad():
        for chunk in chunks:
            ed = st + chunk.shape[1]
            llm_model(input_ids=chunk, use_cache=True, past_key_values=past_key_values)
            st = ed
    
    # ----- Auto Regressive生成 ----- #
    input_ids = input_ids[:, -1:]
    attention_mask = torch.ones(1, ed, dtype=torch.long, device=llm_model.device)
    try:
        for _ in range(max_seq_len):
            with torch.no_grad():
                # ----- Update position ----- #
                ed += 1

                # ----- Update model kwargs ----- #
                # cache_position = torch.arange(ed-1, ed, dtype=torch.long, device = llm_model.device)
                cache_position = torch.arange(past_key_values.get_seq_length(layer_idx=0)-1, 
                                              past_key_values.get_seq_length(layer_idx=0), 
                                              dtype=torch.long, 
                                              device = llm_model.device)
                # ----- 生成token ----- #
                outputs = llm_model(input_ids=input_ids, use_cache=True, past_key_values=past_key_values, cache_position=cache_position)
                logits = outputs.logits
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                token_id = next_token.item()
                input_ids = next_token

                # ----- 判斷是否終止 ----- #
                if token_id in eos_token_ids:
                    break

                # ----- 紀錄token ----- #
                res += [tokenizer.decode(token_id)]
                
                # ----- 輸出文字字串 ----- #
                # print(res[-1], end="", flush=True)
    except:
        for item in ("input_ids", "outputs", "ogits", "next_token", "token_id"):
            try:
                eval(f"del {item}")
            except:
                pass
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
    return "".join(res), past_key_values

In [20]:
import time
i = 0
for sentences in ("半導體廠務通常在做什麼", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理，它裡面有很多關於統計觀念，也請一一闡述", "什麼是巨單交易", "什麼是機器學習"):
    s = time.time()
    _text, _cache = gemma3_resp(sentences, 1)
    print("Spend:", time.time() - s)
    print("\n","-"*50)

Spend: 0.48130178451538086

 --------------------------------------------------
Spend: 0.2872765064239502

 --------------------------------------------------
Spend: 0.16899943351745605

 --------------------------------------------------
Spend: 0.17196917533874512

 --------------------------------------------------


In [17]:
text1, cache1 = gemma3_resp("半導體廠務通常在做什麼")

29
tensor([27], device='cuda:0')
tensor([28], device='cuda:0')
tensor([29], device='cuda:0')
tensor([30], device='cuda:0')
tensor([31], device='cuda:0')
tensor([32], device='cuda:0')
tensor([33], device='cuda:0')
tensor([34], device='cuda:0')
tensor([35], device='cuda:0')
tensor([36], device='cuda:0')
tensor([37], device='cuda:0')
tensor([38], device='cuda:0')
tensor([39], device='cuda:0')
tensor([40], device='cuda:0')
tensor([41], device='cuda:0')
tensor([42], device='cuda:0')


In [18]:
text1

'半導體廠務通常在做以下幾個核心任務：\n\n1.'

In [ ]:
_, cache1 = gemma3_resp("半導體廠務通常在做什麼")

半導體廠務通常在做以下幾個核心任務：

1.

In [ ]:
_, cache2 = gemma3_resp("什麼是AI ? 一句話介紹一下")

AI (Artificial Intelligence) 是一種人工智能 (Artificial Intelligence) 的一種方法，它通過使用计算机 (c-a-I) 模拟人类的智能，从而学习和模仿人类的语言、动作、学习和推理能力。

AI 是一種高度智能的機器學習 (Machine Learning

In [ ]:
import torch
import numpy as np
import torch.nn.functional as F
from typing import List, Dict, Any
from transformers import DynamicCache

def KVCache_merge(caches: List[DynamicCache]):
    results = DynamicCache()
    # ----- 檢查是否全部都為空 ----- #
    empty_check = [c.key_cache[0] is None for c in caches]
    if np.all(empty_check):
        return results

    # ----- 取出layer ----- #
    seq_len = max(c.get_seq_length(layer_idx=0) for c, is_empty in zip(caches, empty_check) if not is_empty)
    first_non_empty_cache = next(c for c, is_empty in zip(caches, empty_check) if not is_empty)
    n_layers = len(first_non_empty_cache.key_cache)
    n_heads, hid_dim = first_non_empty_cache.key_cache[0].shape[1], first_non_empty_cache.key_cache[0].shape[3]
    
    # ----- 建立cache ----- #
    for i in range(n_layers):
        # ----- 依照不同layer去建立cache ----- #
        keys, values = list(), list()
        for c in caches:
            if c.key_cache[0] is None:
                # ----- 如果是空的，則全部補0 ----- #
                key_tensor = torch.zeros((1, n_heads, seq_len, hid_dim), dtype=torch.float32)
                value_tensor = torch.zeros((1, n_heads, seq_len, hid_dim), dtype=torch.float32)
                keys += [key_tensor]
                values += [value_tensor]
                continue
            key_tensor = c.key_cache[i]
            value_tensor = c.value_cache[i]

            # ----- 過長的部分做padding ----- # 
            curr_seq_len = key_tensor.shape[2]
            if curr_seq_len < seq_len:
                padding_to_add = seq_len - curr_seq_len
                key_tensor = F.pad(key_tensor, (0, 0, padding_to_add, 0), "constant", 0)
                value_tensor = F.pad(value_tensor, (0, 0, padding_to_add, 0), "constant", 0)

            keys += [key_tensor]
            values += [value_tensor]
        
        # ----- merge tensor ----- #
        key_batch = torch.cat(keys, dim=0)
        value_batch = torch.cat(values, dim=0)

        # ----- update cache ----- #
        results.update(key_states=key_batch, value_states=value_batch, layer_idx=i)
    results.seen_tokens = seq_len
    return results

In [ ]:
CACHE = KVCache_merge([cache1, cache2])

In [ ]:
CACHE.get_seq_length(layer_idx=0)

169

In [ ]:
CACHE.key_cache[0][:, :, :, 0:1]

tensor([[[[ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],


In [ ]:
def KVCache_split(cache: DynamicCache):
    # ----- 把cache的layer跟數量定義出來 ----- #
    batch_size = cache.key_cache[0].shape[0]
    n_layers = len(cache.key_cache)

    # ----- return的結果 ----- #
    results: List[DynamicCache] = [DynamicCache() for _ in range(batch_size)]

    # ----- by batch操作
    for i in range(batch_size):
        # ----- cache的原始長度，只要用第0層來找即可 ----- #
        """
        因為padding是用0填充，所以如果 hid_dim 和 n_head 都是0，那該位必定padding
        最終找到最後一個非零位置
        """
        sample_key_tensor = cache.key_cache[0][i:i+1] # (1, n_heads, seq_len, hid_dim)
        sum_abs = torch.abs(sample_key_tensor).sum(dim=(1, 3)).squeeze(0)
        non_zero_indices = torch.where(sum_abs > 1e-6)[0] # (seq_len, )

        # ----- seq_len 儲存長度計算 ----- #
        if len(non_zero_indices) == 0: original_seq_len = 0
        else: original_seq_len = non_zero_indices.min().item()
            
        for layer_idx in range(n_layers):
            key_slice = cache.key_cache[layer_idx][i:i+1]
            value_slice = cache.value_cache[layer_idx][i:i+1]

            truncated_key = key_slice[:, :, original_seq_len:, :]
            truncated_value = value_slice[:, :, original_seq_len:, :]
            
            results[i].update(
                key_states=truncated_key,
                value_states=truncated_value,
                layer_idx=layer_idx
            )
        
        # 6. 更新這個 cache 的 seen_tokens
        results[i].seen_tokens = original_seq_len

    return results

In [ ]:
_cache1, _cache2 = KVCache_split(CACHE)
_cache1.key_cache[0][:, :, :, 0:1]

tensor([[[[ 0.0347],
          [-2.8125],
          [-3.2812],
          [-0.4941],
          [ 0.1016],
          [ 1.5000],
          [ 0.8594],
          [-4.0625],
          [-3.6094],
          [-1.4531],
          [ 3.5625],
          [ 4.5312],
          [ 1.5156],
          [-1.7109],
          [-0.5078],
          [-2.1250],
          [-0.5391],
          [-0.0435],
          [ 2.9062],
          [-0.9453],
          [ 0.1157],
          [-0.4453],
          [-0.0967],
          [ 2.5156],
          [ 0.2139],
          [ 0.7148],
          [-2.4219],
          [-3.1719],
          [-1.1016],
          [ 1.7188],
          [ 3.3906],
          [ 1.2656],
          [-1.9922],
          [-0.1641]]]], device='cuda:0', dtype=torch.bfloat16)

In [ ]:
cache1.key_cache[0][:, :, :, 0:1]

tensor([[[[ 0.0347],
          [-2.8125],
          [-3.2812],
          [-0.4941],
          [ 0.1016],
          [ 1.5000],
          [ 0.8594],
          [-4.0625],
          [-3.6094],
          [-1.4531],
          [ 3.5625],
          [ 4.5312],
          [ 1.5156],
          [-1.7109],
          [-0.5078],
          [-2.1250],
          [-0.5391],
          [-0.0435],
          [ 2.9062],
          [-0.9453],
          [ 0.1157],
          [-0.4453],
          [-0.0967],
          [ 2.5156],
          [ 0.2139],
          [ 0.7148],
          [-2.4219],
          [-3.1719],
          [-1.1016],
          [ 1.7188],
          [ 3.3906],
          [ 1.2656],
          [-1.9922],
          [-0.1641]]]], device='cuda:0', dtype=torch.bfloat16)

In [5]:
import torch
tensor=torch.tensor([1,2,3,4,0,0,0])
where = torch.where(tensor==0)[0]
tensor[:(where - len(tensor))[0]]

tensor([1, 2, 3, 4])

In [6]:
tensor.unsqueeze(1)

tensor([[1],
        [2],
        [3],
        [4],
        [0],
        [0],
        [0]])